# LettuceDetect baseline — Hallucination Detection in Tool Calling

Evaluates the off-the-shelf `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint on three test splits we built from ToolACE, each corresponding to one hallucination type:

- **Type 1** — *Hallucination*: answer contradicts tool output.
- **Type 2** — *Overgeneration*: answer adds facts not in tool output.
- **Type 3** — *Missing tool*: answer proposes an action requiring an unavailable tool.

For each split we report span-level (micro + macro F1) and example-level (P/R/F1) metrics.

**Requires a GPU runtime (T4 / P100 / L4).** Inference on ~150 samples takes ~30 s on T4.

Upload the test files (`test.jsonl`, `type2_test.jsonl`, `type3_test.jsonl`) as a Kaggle dataset and point `DATA_DIR` to its mount path below.


In [1]:
import subprocess, sys, os, importlib.util

NEEDED = ("lettucedetect",)
missing = [p for p in NEEDED if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    # Colab pre-loads older numpy/transformers; restart the kernel so new versions are picked up.
    if "google.colab" in sys.modules:
        print("Restarting kernel to pick up new packages — re-run the notebook from the top after restart.")
        os.kill(os.getpid(), 9)
else:
    print("lettucedetect already installed.")


lettucedetect already installed.


## Setup

This notebook runs in **Google Colab** or **Kaggle**. Steps below:

1. **Enable GPU**: `Runtime → Change runtime type → T4 GPU` (Colab) or `Settings → Accelerator → GPU` (Kaggle).
2. **Upload the three test files** (`test.jsonl`, `type2_test.jsonl`, `type3_test.jsonl`):
   - **Colab**: click the folder icon in the left sidebar → upload button → drop the three files into `/content/`. (Or run the upload cell below.)
   - **Kaggle**: upload them as a Kaggle dataset, then point `DATA_DIR` at `/kaggle/input/<your-dataset>/`.
3. **Run all cells**.

The expected filenames are `test.jsonl` (Type 1), `type2_test.jsonl`, `type3_test.jsonl`.

In [2]:
from pathlib import Path

# Auto-detect environment.
CANDIDATE_DIRS = [
    Path("/content"),
    Path("/kaggle/input/halluc-toolace"),
    Path("data"),
    Path("."),
]
REQUIRED = ["type1_test_balanced.jsonl", "type2_test_balanced.jsonl",
            "type3_test_balanced.jsonl", "combined_test.jsonl"]
DATA_DIR = next((d for d in CANDIDATE_DIRS if all((d / f).exists() for f in REQUIRED)), None)

if DATA_DIR is None:
    print("Test files not found.")
    print("Upload: type1_test_balanced.jsonl, type2_test_balanced.jsonl, "
          "type3_test_balanced.jsonl, combined_test.jsonl (via the upload cell below).")
else:
    print(f"Using DATA_DIR = {DATA_DIR}")

MODEL_PATH = "KRLabsOrg/lettucedect-large-modernbert-en-v1"


Using DATA_DIR = /content


### Optional: upload via Colab UI

If you didn't drop the files into `/content/` manually, run this cell — it will pop up a file picker. (Skip if `DATA_DIR` is already set above.)

In [3]:
if DATA_DIR is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_DIR = Path("/content")
        print(f"Uploaded: {list(uploaded.keys())}")
        print(f"DATA_DIR = {DATA_DIR}")
    except ImportError:
        print("Not in Colab. Set DATA_DIR manually or place the files in one of the candidate dirs.")

TEST_FILES = {
    "Combined (all types + clean)":  DATA_DIR / "combined_test.jsonl",
    "Type 1 (Hallucination)":        DATA_DIR / "type1_test_balanced.jsonl",
    "Type 2 (Overgeneration)":       DATA_DIR / "type2_test_balanced.jsonl",
    "Type 3 (Missing Tool)":         DATA_DIR / "type3_test_balanced.jsonl",
}
for name, path in TEST_FILES.items():
    print(f"  {name}: {path}  exists={path.exists()}")


  Combined (all types + clean): /content/combined_test.jsonl  exists=True
  Type 1 (Hallucination): /content/type1_test_balanced.jsonl  exists=True
  Type 2 (Overgeneration): /content/type2_test_balanced.jsonl  exists=True
  Type 3 (Missing Tool): /content/type3_test_balanced.jsonl  exists=True


## Data loading and metrics

In [4]:
import json
from dataclasses import dataclass


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def _char_set(spans):
    chars = set()
    for s in spans:
        chars.update(range(int(s["start"]), int(s["end"])))
    return chars


@dataclass
class Metrics:
    precision: float
    recall: float
    f1: float


def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample["hallucination_labels"])
        pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap
        micro_pred += len(pred)
        micro_gold += len(gold)
        if not pred and not gold:
            macro_f1.append(1.0); continue
        p = overlap / len(pred) if pred else 0.0
        r = overlap / len(gold) if gold else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        macro_f1.append(f1)
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    f_ma = sum(macro_f1) / len(macro_f1) if macro_f1 else 0.0
    return Metrics(p_mi, r_mi, f_mi), f_ma


def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample["hallucination_labels"] else 0
        pred = 1 if preds else 0
        if gold == 1 and pred == 1: tp += 1
        elif gold == 1 and pred == 0: fn += 1
        elif gold == 0 and pred == 1: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return Metrics(p, r, f1)


## Load LettuceDetect

This downloads the model (~400 MB, ModernBERT large) on first call.

In [5]:
from lettucedetect.models.inference import HallucinationDetector

detector = HallucinationDetector(method="transformer", model_path=MODEL_PATH)
print(f"Loaded {MODEL_PATH}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Loaded KRLabsOrg/lettucedect-large-modernbert-en-v1


## Inference and metrics

In [6]:
from tqdm.auto import tqdm

def predict(sample):
    raw = detector.predict(
        context=[sample["context"]],
        question=sample["query"],
        answer=sample["output"],
        output_format="spans",
    )
    return [
        {"start": int(s["start"]), "end": int(s["end"]),
         "text": s.get("text", sample["output"][int(s["start"]):int(s["end"])]),
         "confidence": float(s.get("confidence", 0.0))}
        for s in raw
    ]

results = {}
for name, path in TEST_FILES.items():
    if not path.exists():
        print(f"[skip] {name}: {path} not found")
        continue
    samples = read_jsonl(path)
    preds = [predict(s) for s in tqdm(samples, desc=name)]
    span_micro, span_macro_f1 = span_metrics(samples, preds)
    ex = example_metrics(samples, preds)
    results[name] = {
        "n": len(samples),
        "span_micro": span_micro,
        "span_macro_f1": span_macro_f1,
        "example": ex,
        "preds": preds,
        "samples": samples,
    }
    print(f"\n{name}: span micro P/R/F1 = {span_micro.precision:.3f} / {span_micro.recall:.3f} / {span_micro.f1:.3f}"
          f" | span macro F1 = {span_macro_f1:.3f}"
          f" | example P/R/F1 = {ex.precision:.3f} / {ex.recall:.3f} / {ex.f1:.3f}")


Combined (all types + clean):   0%|          | 0/599 [00:00<?, ?it/s]

W0521 16:03:38.115000 2145 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode



Combined (all types + clean): span micro P/R/F1 = 0.515 / 0.918 / 0.660 | span macro F1 = 0.629 | example P/R/F1 = 0.871 / 0.902 / 0.886


Type 1 (Hallucination):   0%|          | 0/300 [00:00<?, ?it/s]


Type 1 (Hallucination): span micro P/R/F1 = 0.077 / 0.669 / 0.137 | span macro F1 = 0.440 | example P/R/F1 = 0.661 / 0.780 / 0.716


Type 2 (Overgeneration):   0%|          | 0/300 [00:00<?, ?it/s]


Type 2 (Overgeneration): span micro P/R/F1 = 0.570 / 0.998 / 0.726 | span macro F1 = 0.746 | example P/R/F1 = 0.714 / 1.000 / 0.833


Type 3 (Missing Tool):   0%|          | 0/299 [00:00<?, ?it/s]


Type 3 (Missing Tool): span micro P/R/F1 = 0.461 / 0.847 / 0.597 | span macro F1 = 0.671 | example P/R/F1 = 0.697 / 0.926 / 0.795


## Summary table

In [7]:
import pandas as pd

rows = []
for name, r in results.items():
    rows.append({
        "Split": name,
        "N": r["n"],
        "Span micro P": round(r["span_micro"].precision, 3),
        "Span micro R": round(r["span_micro"].recall, 3),
        "Span micro F1": round(r["span_micro"].f1, 3),
        "Span macro F1": round(r["span_macro_f1"], 3),
        "Ex P": round(r["example"].precision, 3),
        "Ex R": round(r["example"].recall, 3),
        "Ex F1": round(r["example"].f1, 3),
    })
df = pd.DataFrame(rows)
df


,Split,N,Span micro P,Span micro R,Span micro F1,Span macro F1,Ex P,Ex R,Ex F1
0,Combined (all types + clean),599,0.515,0.918,0.660,0.629,0.871,0.902,0.886
1,Type 1 (Hallucination),300,0.077,0.669,0.137,0.440,0.661,0.780,0.716
2,Type 2 (Overgeneration),300,0.570,0.998,0.726,0.746,0.714,1.000,0.833
3,Type 3 (Missing Tool),299,0.461,0.847,0.597,0.671,0.697,0.926,0.795


## Qualitative inspection

Show a few samples per split with gold span vs predicted spans.

In [8]:
def show(sample, preds, n=3, width=120):
    out = sample["output"]
    gold = sample["hallucination_labels"]
    print(f"--- {sample.get('id', '?')} ---")
    for g in gold:
        s, e = g["start"], g["end"]
        print(f"  GOLD: [{s}..{e}] {out[s:e]!r}")
    if preds:
        for p in preds:
            s, e = p["start"], p["end"]
            print(f"  PRED: [{s}..{e}] {out[s:e]!r}  (conf={p.get('confidence', 0):.2f})")
    else:
        print(f"  PRED: (none)")
    print()

for name, r in results.items():
    print(f"\n=== {name} ===")
    for i in range(min(3, r["n"])):
        show(r["samples"][i], r["preds"][i])



=== Combined (all types + clean) ===
--- toolace-00899_v0 ---
  GOLD: [85..88] '418'
  PRED: [84..90] ' 418°C'  (conf=0.63)
  PRED: [143..147] ' kWh'  (conf=0.60)
  PRED: [149..370] 'This optimization could lead to a cost savings of approximately 10%. It is advisable to implement these changes along with the recommended actions to reduce the likelihood of failure and achieve more efficient operations.'  (conf=0.96)

--- toolace-00395_v0 ---
  GOLD: [131..144] 'Mercedes-Benz'
  PRED: (none)

--- toolace-00950_v0 ---
  GOLD: [77..88] 'Bob Johnson'
  PRED: [0..71] 'I found two family law lawyers available from 2023-01-24 to 2023-01-25:'  (conf=0.94)
  PRED: [72..90] '1. **Bob Johnson**'  (conf=0.77)
  PRED: [99..105] ' alice'  (conf=0.55)
  PRED: [236..303] 'You can contact either of them to attend the hearing on 2023-01-25.'  (conf=0.91)


=== Type 1 (Hallucination) ===
--- toolace-00899_v0 ---
  GOLD: [85..88] '418'
  PRED: [84..90] ' 418°C'  (conf=0.63)
  PRED: [143..147] ' kWh'  (con